# DPO: Direct Preference Optimization - 실습 코드 1: DPO 학습 (TRL)

- Tutorial ID: `expand-dpo`
- Tutorial: DPO: Direct Preference Optimization
- Section ID: `expand-dpo-code-1`
- Section: 실습 코드 1: DPO 학습 (TRL)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: DPO 학습 (TRL)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#   2) 선호쌍 chosen/rejected가 loss와 policy update 신호로 바뀌는 흐름 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from trl import DPOTrainer, DPOConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# 모델 로드
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")
ref_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")

# 선호도 데이터
from datasets import Dataset
data = {
    "prompt": ["Write a poem", "Explain ML"],
    "chosen": ["Beautiful roses bloom...", "ML is learning from data..."],
    "rejected": ["Roses are red...", "ML is like a computer..."],
}
dataset = Dataset.from_dict(data)

# DPO 학습
training_args = DPOConfig(
    output_dir="./dpo-model",
    beta=0.1,               # KL 페널티 계수
    per_device_train_batch_size=4,
    learning_rate=5e-7,
    num_train_epochs=3,
)
trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=dataset,
)
trainer.train()